# Model Evaluation: Predicting Conflict Escalation

* **Model A:** Baseline (Tabular ACLED + Food + Rain)
* **Model B:** Baseline + Text embeddings

There are four different versions of Model B evaluated throughtout this notebook.
* All events, no PCA (~790 features) `all_nopca`
* All events, PCA (~73 features) `all_pca`
* Conflict-only events, no PCA (~790 features) `conflict_nopca`
* Conflict-only events, PCA (~47 features) `conflict_pca`

This notebook compares the results from the best models where `k`=1.75 and the threshold fix has been applied, as dicussed in the methodology decisions notebook.

Each model configuration has a number of recorded results on:
* Train (2018-2022) - period of time where there is no civil war. Results on training-CV splits. 
* Onset (2023) - including the three months where civil war esclated (April 2023)
* Active (2024-2025) - full period of time with ongoing active civil war

Results are read in from:
`sudan_resuluts.csv` - evaluation results file.
`sudan_results_seeds.xlxs` - file used to test different hyperparameters to stress test findings.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from models.run_best_models import run_best_models
from utils.reporting import read_model_reports

MODELS = [
    "Model A",
    "Model B (conflict-only text PCA)",
    "Model B (conflict-only text non-PCA)",
    "Model B (all-event text non-PCA)",
    "Model B (all-event text PCA)",
]

In [2]:
# Reading in data
def apply_final_config(df: pd.DataFrame, config) -> pd.DataFrame:
    """Filter the results for the decided config."""
    mask = pd.Series(True, index=df.index)
    for col, val in config.items():
        mask &= df[col] == val

    food_ok = (df["include_food"] == False) | (df["price_recency"] == True)
    mask &= food_ok
    return df[mask].copy()

def variant_label(row):
    if row["include_text"] == False:
        return "model_a"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == True:
        return "conflict_pca"
    if row["conflict_only_embeddings"] == True and row["use_pca"] == False:
        return "conflict_nopca"
    if row["conflict_only_embeddings"] == False and row["use_pca"] == True:
        return "all_pca"
    return "all_nopca"


part1_config = {
    "k": 1.75,
    "threshold_fix_applied": True,
    "seed": 23
}


all_results = pd.read_csv("evaluation/sudan_results.csv")
results = apply_final_config(all_results, part1_config)
results["variant"] = results.apply(variant_label, axis=1)

In [17]:
# Baseline summary tables
def baseline_performance_summary(results):
    """Calculates aggregate performance statistics for the Baseline model (Model A) only."""
    baseline = results[results["variant"] == "model_a"].copy()

    metrics = [
        "train_cv_aupr", "train_cv_f1",
        "onset_aupr", "onset_f1_class1",
        "active_aupr", "active_f1_class1"
    ]

    for m in metrics:
        baseline[m] = pd.to_numeric(baseline[m], errors="coerce")

    summary = baseline[metrics].agg(["mean", "median", "min", "max", "std"]).round(4).T
    
    summary.insert(0, "n_configs", len(baseline))
    summary.index = summary.index.str.replace("_class1", "")
    
    return summary

def baseline_comparison_table(results, metrics, split):
    recall_col = f"{split}_recall_class1"
    metrics = list(dict.fromkeys(metrics))

    cols_to_convert = metrics + ([recall_col] if recall_col in results.columns else [])
    for m in cols_to_convert:
        results[m] = pd.to_numeric(results[m], errors="coerce")

    key_cols = [
        "include_food",
        "include_rain",
        "k",
        "event_col",
        "n_splits",
        "price_recency",
    ]

    baseline = results[results["include_text"] == False].set_index(key_cols)
    text_df = results[results["include_text"] == True].copy()

    results_comparison = []
    for variant, grp in text_df.groupby("variant"):
        grp = grp.set_index(key_cols)

        join_cols = list(dict.fromkeys(metrics + ([recall_col] if recall_col in grp.columns else [])))
        matched = grp.join(baseline[join_cols], rsuffix="_base", how="inner")

        mean_preds = (
            matched["n_predictors"].mean()
            if "n_predictors" in matched.columns
            else float("nan")
        )

        collapse_rate = (
            (matched[recall_col] > 0.9).mean() * 100
            if recall_col in matched.columns
            else float("nan")
        )

        row = {
            "variant": variant,
            "n_matched_pairs": len(matched),
            "mean_n_predictors": round(mean_preds, 1),
            "percent_collapsed_recall": round(collapse_rate, 1),
        }

        for m in metrics:
            diff = matched[m] - matched[f"{m}_base"]
            row[f"mean_{m}"] = round(matched[m].mean(), 4)
            row[f"mean_{m}_baseline"] = round(matched[f"{m}_base"].mean(), 4)
            row[f"percent_better_baseline_{m}"] = round((diff > 0).mean() * 100, 1)
            row[f"mean_diff_{m}"] = round(diff.mean(), 4)
            row[f"median_diff_{m}"] = round(diff.median(), 4)

        results_comparison.append(row)

    results_table = pd.DataFrame(results_comparison).set_index("variant")
    display(results_table)
    return


# Recall/precision table
def recall_precision_summary(results, split):
    recall_col = f"{split}_recall_class1"
    precision_col = f"{split}_precision_class1"

    results_rp = results.copy()
    results_rp["collapsed"] = results_rp[recall_col] > 0.9

    return (
        results_rp.groupby("variant")
        .agg(
            n_configs=(recall_col, "count"),
            mean_recall=(recall_col, "mean"),
            mean_precision=(precision_col, "mean"),
            collapse_rate=("collapsed", "mean"),
        )
        .round(3)
    )

In [18]:
# Table for comparing event columns
def event_col_preference_table(results, metrics):
    match_cols = ["include_food", "include_rain", "n_splits", "variant"]

    sub = results[results["event_col"] == "sub_event_type"].set_index(match_cols)
    evt = results[results["event_col"] == "event_type"].set_index(match_cols)

    rows = []
    for metric in metrics:
        paired = sub[[metric]].join(
            evt[[metric]], lsuffix="_sub", rsuffix="_evt", how="inner"
        )
        for variant, grp in paired.reset_index().groupby("variant"):
            n_sub_wins = (grp[f"{metric}_sub"] > grp[f"{metric}_evt"]).sum()
            n_evt_wins = (grp[f"{metric}_evt"] > grp[f"{metric}_sub"]).sum()
            n_total = len(grp)
            rows.append(
                {
                    "variant": variant,
                    "metric": metric,
                    "n_pairs": n_total,
                    "sub_event_type_wins": n_sub_wins,
                    "event_type_wins": n_evt_wins,
                }
            )

    table = pd.DataFrame(rows)

    totals = table.groupby("metric")[
        ["n_pairs", "sub_event_type_wins", "event_type_wins"]
    ].sum()
    totals["variant"] = "TOTAL"
    totals = totals.reset_index()
    print(totals)

    return pd.concat([table, totals], ignore_index=True)

# Introduction
Predicting the exact outbreak of a rare, unprecedented civil war is an inherently difficult forecasting task. Before evaluating the impact of text embeddings, it is important to establish the predictive baseline of Model A, which relies purely on structural data: historical tabular ACLED counts, food prices, and rainfall.

Across 16 tested configurations Model A has a **mean train-CV AUPR of 0.2388** and **F1 of 0.3408**.

While these absolute metrics are lower than standard machine learning benchmarks, this reflects both the limited scope of this project and the challenge of training data in conflict prediction. As dicussed in the main project report, comparable conflict forecasting models take a multi-country approach which both expands the available training data (including available structural variables) and the number of conflict escalations for the model to learn from. The training data, while it includes conflict escalations it does not include the type of esclation it is trying to predict. The objective of this evaluation project is not to present a flawless predictive system, but to determine whether text embeddings can overcome the limitations of structural data to provide an earlier, measurable warning signal.

In [19]:
baseline_performance_summary(results)

,n_configs,mean,median,min,max,std
train_cv_aupr,16,0.2388,0.2356,0.2154,0.2724,0.0163
train_cv_f1,16,0.3408,0.3366,0.3184,0.3683,0.0180
onset_aupr,16,0.3198,0.3138,0.2700,0.3738,0.0316
onset_f1,16,0.3224,0.3212,0.2192,0.4000,0.0424
active_aupr,16,0.2267,0.2218,0.1899,0.2785,0.0213
active_f1,16,0.2586,0.2564,0.2235,0.2985,0.0240


# Part 1 - does text improve model performance?
Part 1 looks at the question of 'does text improve performance?' rather than comparing the final chosen best models. The following things have been fixed (see methdodology decisions for discussion):

* `k`=1.75 - this is fixed as it defines the prediction target.
* The threshold fix has been applied - this was a bug fix that has been resolved. 
* Food price recency flag has been included - original runs (not included in these results) did not include the price-recency flag for food price data and instead relied on forward fill. More on how zero values were filled is dicussed in the methodology notebook.

The following implementation choices are allowed to roam in part 1 as they are do not change the underlying task, only what inputs the model draws on and how it is fit:
* `n_splits`
* `event_col`
* The inclusion of additional strucutral variables (food/rain)

# 1.1 Training performance
The results on the cross-validation training splits give an indication on whether the models are overfitting to the training data and would therefore not be generalisable.

As the aim is to understand if adding text improves the baseline (structural features only), each baseline and text-added configuration was compared on training-CV AUPR and F1. In the initial evaluation stage, the percetange of times the text model (matched to the same baseline configuration) beats baseline is evaluated. 

Evaluating model performance across the peaceful baseline period reveals that *text embeddings degrade model performance relative to structural features*. On AUPR, only two of the four variants ever beat baseline, and only barely (four models total).

* Neither all-event (`all_nopca` and `all_pca`) variant beats baseline on AUPR in any comparison. 
* Conflict-only text, both with (`conflict_pca`) and without PCA (`conflict_nopca`), is tied as the best performing model on training AUPR, each beating baseline in 2 of 16 comparisons (12.5%). 
* On F1, conflict-only text without PCA (`conflict_nopca`) is the clearer of the two, beating baseline in 4 of 16 comparisons (25%) against conflict-only PCA's 2 of 16 (12.5%), so it's the stronger overall performer of the two once both metrics are considered, but the AUPR result on its own does not distinguish between them.
  
In a standard machine learning pipeline, discarding a model that loses across 100% of training folds would be best practice.

**Baseline vs best performing model on train-cv**

| Model Variant | Mean Train CV AUPR | Mean Train CV F1 | AUPR Win Rate vs. Baseline | F1 Win Rate vs. Baseline |
| :--- | :---: | :---: | :---: | :---: |
| **Model A (`model_a`)** | **0.2388** | **0.3408** | — | — |
| **Model B (`conflict_nopca`)** | 0.2267 | 0.3235 | 12.5% | 25.0% |




In [20]:
# Baseline comparison for training AUPR and F1
baseline_comparison_table(results, ["train_cv_aupr", "train_cv_f1"], "train")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_train_cv_aupr,mean_train_cv_aupr_baseline,percent_better_baseline_train_cv_aupr,mean_diff_train_cv_aupr,median_diff_train_cv_aupr,mean_train_cv_f1,mean_train_cv_f1_baseline,percent_better_baseline_train_cv_f1,mean_diff_train_cv_f1,median_diff_train_cv_f1
variant,,,,,,,,,,,,,
all_nopca,16,790.5,NaN,0.1993,0.2388,0.0,-0.0395,-0.0374,0.2881,0.3408,0.0,-0.0526,-0.0473
all_pca,16,72.5,NaN,0.2049,0.2388,0.0,-0.0339,-0.0368,0.2943,0.3408,0.0,-0.0465,-0.0431
conflict_nopca,16,790.5,NaN,0.2267,0.2388,12.5,-0.0121,-0.0130,0.3235,0.3408,25.0,-0.0173,-0.0184
conflict_pca,16,46.5,NaN,0.2255,0.2388,12.5,-0.0133,-0.0160,0.3225,0.3408,12.5,-0.0183,-0.0182


# 1.2 Onset performance


During the 2023 onset period, text features do provide a performance boost over baseline, but identifying the "best" text variant depends on whether evaluation prioritises AUPR or F1. This metric split aligns strictly with corpus selection (conflict-only vs. all-events) rather than dimensionality reduction (PCA).

*Conflict-only text*

Both conflict-only variants outperform both all-event variants on AUPR:
* Conflict-only without PCA (`conflict_nopca`): 75.0% of 16 matched pairs modestly beat baseline. With a mean AUPR diff of +0.017 above baseline.
* Conflict-only with PCA (`conflict_pca`): 68.8% of 16 models beat baseline. With +0.009 mean AUPR diff above baseline. 
In this model's context, AUPR demonstrates how well the model balances the accuracy of the region-months it flags as escalations (precision) and its ability to find the actual escalations (recall) across all confidence thresholds. 


*All-event text*

In contrast both all-event variants outperform both conflict-only variants on F1:
* All-event non-PCA (`all_nopca`): Train-CV F1 of 87.5% of baseline models, mean diff +0.032 above baseline
* All-event PCA (`all_pca`): Train-CV F1 of 43.8%, mean diff -0.005 below baseline. 
The shows the all-event model assigns higher esclation probabilities to pre-conflict regions which causes recall to increase, but it lacks precision. The all-event model assigns higher escalation probabilities to pre-conflict regions. At the calibrated F1 threshold, this causes recall to surge—jumping from ~25% in conflict-only models up to 75%–88% in the all-event model. Because conflict-only models are too conservative and miss most onset events (low recall cripples their F1), the massive recall boost in all_nopca easily outweighs its moderate penalty in precision, pushing its overall F1 score significantly higher.

All-event text with PCA is the weakest text variant on AUPR specifically. Though it catches more escalations (better recall), it's precicision is the lowest of the models. It is  the only one with a negative mean difference on AUPR (-0.0085), and the only one that fails to beat baseline in a majority of comparisons on either metric.

| Model Variant | AUPR Win Rate | Mean AUPR Diff | F1 Win Rate | Mean F1 Diff | Mean Recall | Mean Precision | Collapse Rate |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model A (Baseline)** | — | — | — | — | 38.4% | 29.3% | **0.0%** |
| **`conflict_nopca`** | **75.0%** | **+0.0174** | 25.0% | -0.0217 | 25.0% | **45.4%** | **0.0%** |
| **`conflict_pca`** | 68.8% | +0.0090 | 12.5% | -0.0308 | 25.6% | 40.5% | **0.0%** |
| **`all_nopca`** | 56.2% | +0.0038 | **87.5%** | **+0.0318** | **75.0%** | 23.5% | 12.5% |
| **`all_pca`** | 43.8% | -0.0085 | 43.8% | -0.0050 | 42.5% | 29.2% | 6.2% |

The difference between this variation isn't two different directions by chance. Conflict-only text runs a much more conservative operating point, it ranks escalation months well overall (hence the stronger AUPR across both its PCA and non-PCA forms), but is cautious about calling any specific month an escalation. All-event text runs the opposite, flagging more broadly, which drives its F1 advantage but also means two of all-event non-PCA's 16 configurations collapse to predicting esclations for almost everything (`onset_recall_class1 > 0.9`). Given that a missed escalation is treated as the more costly error for an early-warning system in this project's framing, all-event text's recall-leaning approach is not obviously the wrong choice, but it should be considered a trade-off.


In [8]:
onset_metrics = ["onset_aupr", "onset_f1_class1"]

# Onset comparison to baseline
baseline_comparison_table(results, onset_metrics, "onset")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,percent_better_baseline_onset_aupr,mean_diff_onset_aupr,median_diff_onset_aupr,percent_better_baseline_onset_f1_class1,mean_diff_onset_f1_class1,median_diff_onset_f1_class1
variant,,,,,,,,,
all_nopca,16,790.5,12.5,56.2,0.0038,0.0063,87.5,0.0318,0.0310
all_pca,16,72.5,6.2,43.8,-0.0085,-0.0054,43.8,-0.0050,-0.0022
conflict_nopca,16,790.5,0.0,75.0,0.0174,0.0202,25.0,-0.0217,-0.0362
conflict_pca,16,46.5,0.0,68.8,0.0090,0.0038,12.5,-0.0308,-0.0385


In [21]:
# Onset precision and recall
recall_precision_summary(results, "onset")

,n_configs,mean_recall,mean_precision,collapse_rate
variant,,,,
all_nopca,16,0.750,0.235,0.125
all_pca,16,0.425,0.292,0.062
conflict_nopca,16,0.250,0.454,0.000
conflict_pca,16,0.256,0.405,0.000
model_a,16,0.384,0.293,0.000


**Removing collapsed models**

To test whether the performance gains of `all_nopca` were artificially driven by the 12.5% threshold collapse rate, the matched baseline comparison was re-run on  non-collapsed configurations (`onset_recall < 0.9`).

Both all-event text variants sit above baseline on recall and below it on precision. Both conflict-only text variants sit below baseline on recall and above it on precision. 

`all_nopca` is the extreme case at both ends. It has nearly double baseline's recall, and the only variant with a high collapse rate (12.5%) when setting the collapse threshold to `recall > 0.9`. `all_pca` shows recall in the same direction but far more mildly (recall 0.425 vs baseline's 0.384, a modest lean rather than a strong one), and its 6.2% collapse rate (1 of 16 configs) it noteworthy but not the dominant outcome. 

In comparison to the baseline:

* **F1** `all_nopca` retains its F1 advantage, beating the structural baseline in **85.7% of matched pairs** (12/14 runs) with a mean F1 gain of **+0.0299** (median **+0.0310**).
* **Recall** Non-collapsed `all_nopca` recall settles at **72.6%**, remaining nearly 2x higher than the baseline (38.2%) and 3x higher than conflict-only text (25.0%).
* **AUPR Trade-off:** As expected, `all_nopca` mean AUPR difference becomes neutral (**-0.0007**, median **+0.0010**), reinforcing that `conflict_nopca` remains the optimal variant for threshold-agnostic probability ranking, while `all_nopca` provides the superior discrete early-warning alarm.

In [10]:
# Recall and precision for non-collapsed models 
results_no_collapse = results[results["onset_recall_class1"] < 0.9].copy()
recall_precision_summary(results_no_collapse, "onset")

,n_configs,mean_recall,mean_precision,collapse_rate
variant,,,,
all_nopca,14,0.726,0.237,0.0
all_pca,15,0.390,0.296,0.0
conflict_nopca,16,0.250,0.454,0.0
conflict_pca,16,0.256,0.405,0.0
model_a,16,0.384,0.293,0.0


In [11]:
# Comparison to baseline for non-collapsed models
baseline_comparison_table(results_no_collapse, onset_metrics, "onset")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,percent_better_baseline_onset_aupr,mean_diff_onset_aupr,median_diff_onset_aupr,percent_better_baseline_onset_f1_class1,mean_diff_onset_f1_class1,median_diff_onset_f1_class1
variant,,,,,,,,,
all_nopca,14,791.8,0.0,50.0,-0.0007,0.0010,85.7,0.0299,0.0310
all_pca,15,73.3,0.0,40.0,-0.0100,-0.0082,40.0,-0.0089,-0.0033
conflict_nopca,16,790.5,0.0,75.0,0.0174,0.0202,25.0,-0.0217,-0.0362
conflict_pca,16,46.5,0.0,68.8,0.0090,0.0038,12.5,-0.0308,-0.0385


# 1.3 Active performance

*Once the war is underway, the baseline model prevails almost everywhere*. All four text variants now show negative mean AUPR differences against baseline.

Three of the four text variants perform very poorly during active conflict, and all show negative mean differences on both AUPR and F1. Once conflict has been running for months, region-month event counts stop being zero-inflated and the autoregressive/structural features are doing the actual work. The text embeddings aren't acting as a precursor signal the way they might pre-escalation, they're mostly extra dimensions that can cause overfitting.

During active conflict, the text model all-text with PCA is the best performing against baseline. During active conflict dimensionality reduction appears to specifically help text remain useful once conflict is underway. However, the same compression only works slightly on the conflict-only text model during active war. The pattern suggests it isn't PCA alone or text alone driving the active-period result, but the combination of the full event corpus (not just conflict events) compressed down to a smaller, less overfitting-prone feature set.

| Model Variant | Mean Active AUPR | AUPR Win Rate vs. Base | Mean Active F1 | F1 Win Rate vs. Base | Mean Recall | Mean Precision |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Model A (Baseline)** | **0.2267** | — | **0.2586** | — | 41.4% | **20.8%** |
| **`all_pca`** | 0.2180 | 25.0% | 0.2453 | **50.0%** | 43.4% | 20.4% |
| **`conflict_pca`** | 0.1894 | 6.2% | 0.1397 | 6.2% | 15.8% | 20.6% |
| **`conflict_nopca`** | 0.1672 | 0.0% | 0.1028 | 0.0% | 10.7% | 18.3% |
| **`all_nopca`** | 0.1558 | **0.0%** | 0.2399 | 25.0% | **83.5%** | 14.1% |

In [22]:
active_metrics = ["active_aupr", "active_f1_class1"]
baseline_comparison_table(results, active_metrics, "active")

,n_matched_pairs,mean_n_predictors,percent_collapsed_recall,mean_active_aupr,mean_active_aupr_baseline,percent_better_baseline_active_aupr,mean_diff_active_aupr,median_diff_active_aupr,mean_active_f1_class1,mean_active_f1_class1_baseline,percent_better_baseline_active_f1_class1,mean_diff_active_f1_class1,median_diff_active_f1_class1
variant,,,,,,,,,,,,,
all_nopca,16,790.5,56.2,0.1558,0.2267,0.0,-0.0708,-0.0703,0.2399,0.2586,25.0,-0.0187,-0.0097
all_pca,16,72.5,12.5,0.2180,0.2267,25.0,-0.0087,-0.0107,0.2453,0.2586,50.0,-0.0133,0.0032
conflict_nopca,16,790.5,0.0,0.1672,0.2267,0.0,-0.0595,-0.0594,0.1028,0.2586,0.0,-0.1557,-0.1666
conflict_pca,16,46.5,0.0,0.1894,0.2267,6.2,-0.0373,-0.0359,0.1397,0.2586,6.2,-0.1188,-0.0960


In [ ]:
#TODO inset a chart here

# Part 2 - choosing the best model
For part two, the single best model configuration for the baseline model (Model A) will be comapred against the best model configs for the text variants. In order to make a fair comparison, configurations that were allowed to run freely in part 1 have been limited.

# 2.1 Defining the final config

## 2.1.1 Setting included data

In part 1 dropping rain or dropping food, sometimes scores marginally higher for some variants. Deliberately not following that signal here is the point, if the final comparison quietly adopted whichever ablation flattered each model, the comparison would no longer be about text, it would be about whichever feature set happened to win, model by model. 

For the final model comparison, **food price features are retained while rainfall features are dropped globally**. This decision is grounded in three key empirical and domain findings:
* **AUPR degradation**: Including rainfall data appears to hurt model performance in all models. In the baseline model, including rain lowers train-cv AUPR from 0..2492 to 0.2285.
* **SHAP feature importance**: SHAP feature importance analysis shows that the models heavily downweight rainfall - it accounts for only 6.3%-8.3% of predictive weight in training. 

Conversely, food price commodities and the `months_since_reading_*` price recency indicator are retained because they capture real-time supply chain disruptions and local economic shocks directly tied to conflict escalation. 

In [ ]:
rain_summary = results.groupby(['variant', 'include_rain'])['train_cv_aupr'].mean().round(4)

no_rain = rain_summary.xs(False, level='include_rain')
with_rain = rain_summary.xs(True, level='include_rain')

rain_comparison_table = pd.DataFrame({
    'Train-CV (No Rain) mean AUPR': no_rain,
    'Train-CV (With Rain) mean AUPR': with_rain,
    'TrainCV diff mean AUPR': with_rain - no_rain,

}).round(4)

display(rain_comparison_table)

## 2.1.2 Setting cross-validation folds (`n_splits = 5`)

The `n_splits` parameter determines the number of expanding-window cross-validation folds used during hyperparameter tuning (testing both `n=4`and `n=5` across models). 

Across models, cross-validation training AUPR scores between the two choices are almost identical, showing a small difference of under 0.006 (0.2220 for four splits versus 0.2162 for five splits). Five splits is selected because its structure aligns naturally with the five-year training period (2018–2022), providing intuitive annual expanding increments that maximise training data volume per fold. 

A supporting check (holding food, rain, and event_col fixed and varying only n_splits) found the two all-event text variants are 1.5-1.7x more
sensitive to this choice than the baseline, while conflict-only text and the baseline itself are comparatively stable. Results for all-event text in Part 2 should be read with that in mind.

In [ ]:
metrics = ["train_cv_aupr", "train_cv_f1"]

results.groupby("n_splits")[metrics].agg(["mean", "max", "count"]).round(4)

In [ ]:
results["variant"] = results.apply(variant_label, axis=1)
results.groupby(["n_splits", "variant"])[metrics].agg(["mean", "count"]).round(4)

## 2.1.3 Event type (`event_col = `sub_event_type`)
The `event_col` parameter determines whether ACLED event features are chosen from the six events or the 25 sub-event types. Sub-events naturally give the model greater detail but this level of disaggregation reduces the number of positive instances of that sub-event. 

High-level `event-type` categories maintain denser counts, whereas granular `sub_event_type` categories provide the model with richer tactical detail at the cost of increasing sparsity.

The training-cv results demonstrate `sub_event_type`s slight advantage, yielding higher Train CV AUPR in **21 out of 40 paired comparisons** (52.5%). While pretty much a tie in results, domain-knowledge again is important here. The literature indicates that more granular categories could 

To maximise feature detail and improving performance slightly, `event_col` is set to `sub_event_type` across all baseline and text-augmented configurations. 

In [ ]:
event_col_preference_table(results, ["train_cv_aupr", "onset_aupr"])

#### Therefore we set the best model config to the following:

In [ ]:
part2_config = {
    "k": 1.75,
    "threshold_fix_applied": True,
    "price_recency": True,
    "event_col": "sub_event_type",
    "include_food": True,
    "include_rain": False,
    "n_splits": 5,
}

In [ ]:
# run_best_models(part2_config) # No need to rerun for now
best_model_results, best_model_params, best_model_shap, best_model_onset_pred = read_model_reports(MODELS)

# 2.2 Final model results (averaged across 5 random-search seeds)

All figures below are the mean across the five random-search seeds tested (23, 32, 111, 999, 2025) at the final configuration set in part 2.1. 

**Training AUPR**: None of the text variants beat Model A's mean training AUPR (0.2286), though `conflict_pca` comes the closest at 0.2276. 

**Onset AUPR**: both conflict-only text variants beat Model A on average, `conflict_nopca` at 0.3637 and `conflict_pca` at 0.3566 against Model A's 0.339. `conflict_pca` is the more robust of the two: its lowest score across all five seeds (0.341) still exceeds Model A's mean. `conflict_nopca`'s advantage is real on average but noisier, its seed-to-seed standard deviation (0.028) is larger than its mean advantage over Model A (+0.025).

**Onset F1** (threshold-dependent): Model A leads at 0.384, against a best-of-the-rest 0.369 for `all_pca`. Model A's minimum F1 across the five seeds (0.370) still beats every other model's mean.

In [ ]:
final_results = apply_final_config(all_results, part2_config)
final_results["variant"] = final_results.apply(variant_label, axis=1)

In [ ]:
metric_cols = [
    "train_cv_aupr", "train_cv_f1",
    "onset_aupr", "onset_f1_class1", "onset_recall_class1", "onset_precision_class1",
    "active_aupr", "active_f1_class1", "active_recall_class1", "active_precision_class1"
]

main_metrics_df = final_results.groupby("variant")[metric_cols].mean().reset_index()

main_metrics_df["variant"] = pd.Categorical(main_metrics_df["variant"])

display_df = pd.DataFrame({
    "Model": main_metrics_df["variant"],
    "Mean Train AUPR": main_metrics_df["train_cv_aupr"].round(4),
    "Mean Train F1": main_metrics_df["train_cv_f1"].round(4),
    "Mean Onset AUPR": main_metrics_df["onset_aupr"].round(4),
    "Mean Onset F1": main_metrics_df["onset_f1_class1"].round(4),
    "Mean Onset Recall": (main_metrics_df["onset_recall_class1"] * 100).round(1).astype(str) + "%",
    "Mean Onset Precision": (main_metrics_df["onset_precision_class1"] * 100).round(1).astype(str) + "%",
    "Mean Active AUPR": main_metrics_df["active_aupr"].round(4),
    "Mean Active F1": main_metrics_df["active_f1_class1"].round(4),
    "Mean Active Recall": (main_metrics_df["active_recall_class1"] * 100).round(1).astype(str) + "%",
    "Mean Active Precision": (main_metrics_df["active_precision_class1"] * 100).round(1).astype(str) + "%"
})

display(display_df)

# 2.3 Regional Onset Performance

Evaluating aggregate onset AUPR alone suggests that conflict-only text variants perform strongly. However, this aggregate metric masks critical geographic failures when broken down by region. Because the 2023 escalation was concentrated in specific hotspots, an early-warning system must successfully flag escalation in the regions where fighting actually broke out—most notably Khartoum in April 2023. The table below evaluates performance using the best configuration for each variant, focusing on recall across key escalation regions and Khartoum specifically.

Conflict-only text without PCA is not the best variant here. It misses all escalations in Khartoum entirely and catches only a small fraction of key region escalations overall. All-event text without PCA does the opposite: it beats both the baseline and conflict-only models on regional recall, catching escalation months in Khartoum that none of the other text models flag.

At first glance, all-event text without PCA has the highest regional and Khartoum recall of all models, but this needs to be read against its overall predicted-positive rate, which is far higher than any other model's (0.745, compared to 0.18–0.38 for the other four models). Its key region precision (0.284) is slightly higher than its overall precision (0.240); however, this remains a low level of precision and is the lowest overall.

This reflects the exact same trade-off that shows up in the aggregate onset F1 numbers (where all-event text has the stronger F1 and conflict-only text has the stronger AUPR), just made concrete at the regional level. AUPR measures ranking quality across every possible threshold, meaning a model can score well there while still being too conservative to flag the events that matter most at its actual deployed threshold. The conflict-only models' onset recall sits well below the other models, and that shortfall is most pronounced in the exact regions where the war started.

For an early-warning use case, missing Khartoum isn't a minor flaw—it is fundamentally failing to achieve its core purpose. On that basis, all-event text without PCA is the better candidate for onset detection specifically, even though it is the weaker performer on train-CV and aggregate AUPR. 

However, this finding rests on a very small sample size of actual escalation. Because Sudan's 2023 outbreak is the only event of its kind in this dataset, the claim that text helps at onset should be interpreted strictly as "text helped for this one escalation," rather than as a validated general property.

In [ ]:
KEY_REGIONS = [
    "Khartoum", "North Darfur", "South Darfur", "West Darfur",
    "Central Darfur", "East Darfur", "West Kordofan", "South Kordofan",
]

def region_recall_table(onset_pred_df, regions, model_col="model"):
    rows = []
    for model, grp in onset_pred_df.groupby(model_col):
        key_region_rows = grp[grp["region"].isin(regions)]
        khartoum_rows = grp[grp["region"] == "Khartoum"]

        def recall_precision(sub):
            n_true = sub["y_true"].sum()
            n_pred_pos = sub["y_pred"].sum()
            n_caught = ((sub["y_true"] == 1) & (sub["y_pred"] == 1)).sum()
            recall = n_caught / n_true if n_true else float("nan")
            precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
            return n_caught, n_true, n_pred_pos, recall, precision

        n_caught, n_true, n_pred_pos, key_recall, key_precision = recall_precision(key_region_rows)
        kh_caught, kh_true, kh_pred_pos, kh_recall, kh_precision = recall_precision(khartoum_rows)

        overall_pred_pos_rate = grp["y_pred"].mean()

        rows.append({
            "model": model,
            "overall_pred_positive_rate": round(overall_pred_pos_rate, 3),
            "key_regions_caught": f"{n_caught}/{n_true}",
            "key_regions_recall": round(key_recall, 3),
            "key_regions_precision": round(key_precision, 3),
            "khartoum_caught": f"{kh_caught}/{kh_true}",
            "khartoum_recall": round(kh_recall, 3),
            "khartoum_precision": round(kh_precision, 3),
        })
    return pd.DataFrame(rows).set_index("model")

region_recall_table(best_model_onset_pred, KEY_REGIONS)


# 2.4 Feature importance

SHAP scores help identify why model performance varies across different conflict dynamics. To test whether text’s predictive contribution changes once war is underway, SHAP importance was computed separately for the training (2018–2022), onset (2023), and active conflict (2024–2025) periods.

Rather than diminishing once conflict begins, text embeddings' share of feature importance actually expands during the active war period across all text variants:

* **All-event non-PCA (`all_nopca`):** Text importance rises steadily from 61.6% in training, to 72.3% at onset, and peaks at 75.1% during active conflict.
* **Conflict-only non-PCA (`conflict_nopca`):** Text importance rises from 45.7% in training, to 49.5% at onset, and reaches 55.7% during active conflict.
* **PCA Variants:** Both PCA-compressed variants show a similar upward shift from training into active war, hovering around 56.7% (`all_pca`) and 44.7% (`conflict_pca`).

This mechanism explains the significant drop in active test performance for text models. Instead of deprioritising text features once their precursor early-warning value has passed, the tree models lean more heavily on text embeddings during active war. 

Meanwhile, Model A relies on structural rolling stats (48.6%) and tabular ACLED counts (33.3%), which become highly informative once active violence is established. Because text models allow high-dimensional embeddings to crowd out these updating structural counts, their active-period AUPR and F1 scores suffer a sharp decline.

In [ ]:
#TODO wordcount longer during conflict ?

In [ ]:
shap_by_category = (
    best_model_shap.groupby(["model", "dataset", "category"])["mean_abs_shap"]
    .sum()
    .reset_index()
)

shap_by_category["total_SHAP"] = shap_by_category.groupby(["model", "dataset"])[
    "mean_abs_shap"
].transform("sum")
shap_by_category["% Importance"] = (
    shap_by_category["mean_abs_shap"] / shap_by_category["total_SHAP"] * 100
)

fig = px.bar(
    shap_by_category,
    x="model",
    y="% Importance",
    color="category",
    facet_row="dataset",
    category_orders={"dataset": ["train", "onset", "active"]},
    title="SHAP feature importance by category (Train vs Onset vs Active)",
    text_auto=".1f",
    color_discrete_sequence=px.colors.qualitative.Bold,
    height=900,
)

fig.update_layout(
    xaxis_title="",
    legend_title_text="Feature Category",
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1].capitalize()))

# Add light gridlines and standardise y-axis titles for all rows
fig.update_yaxes(
    title_text="Relative Importance (%)",
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black")

fig.show()

# 3. Another country results to do

# 4. Part 4 - Testing across different parameters
To verify that model rankings and hyperparameter optimization (`RandomizedSearchCV`) are driven by genuine feature signals, the training and evaluation pipeline was evaluated across four additional random search seeds (32, 111, 2025 and 999).

he multi-seed sensitivity analysis confirms that the primary findings are structurally robust.


* **`all_nopca` still wins at onset:** `all_nopca` maintains the highest mean Onset AUPR (0.3517) and lowest seed volatility on Onset F1 (std 0.0113). Even its lowest Onset AUPR score across all seeds (0.3370) exceeds the baseline model's mean score (0.3307).
* **Regional onset stability:** Detection of regional escalation is fairly consistent across seeds. Every 4 of the 5 `all_nopca` seeds hit the exact same 2/5 (40.0%) threshold, with seed 999 performing slightly better at 3/5 (60.0%). No seed ever misses Khartoum entirely.
* **Active-Period Performance Decay:** All text  (`all_nopca`) consistently degrades during active conflict across all seeds (Active AUPR mean 0.1632), whereas PCA compression (`all_pca`) successfully prevents feature overfitting (Active AUPR mean 0.2074, Active F1 mean 0.2592)[cite: 1].

These results confirm that the performance edge of `all_nopca` at onset and `all_pca` during active war reflects genuine predictive value rather than random search optimization noise

In [ ]:
# Testing variance across all seeds
seeds = pd.read_excel("evaluation/sudan_results_seeds.xlsx", sheet_name="results_seeds")
seeds["variant"] = seeds.apply(variant_label, axis=1)
metrics = ["onset_aupr", "onset_f1_class1", "active_aupr", "active_f1_class1"]

for m in metrics:
    seeds[m] = pd.to_numeric(seeds[m], errors="coerce")

seed_summary = seeds.groupby("variant")[metrics].agg(["mean", "std", "min", "max"]).round(4)
seed_summary

In [ ]:
seeds_regional = pd.read_excel("evaluation/sudan_results_seeds.xlsx", sheet_name="onset_predictions_seeds")
regional_seed_summary = region_recall_table(
    seeds_regional, KEY_REGIONS, model_col="model"
)
regional_seed_summary

In [ ]:
all_nopca_df = seeds_regional[
    seeds_regional["model"] == "Model B (all-event text non-PCA)"
]

stability_rows = []
for seed, grp in all_nopca_df.groupby("seed"):
    key_df = grp[grp["region"].isin(KEY_REGIONS)]
    kh_df = grp[grp["region"] == "Khartoum"]

    # Key Regions calculations
    k_true = key_df["y_true"].sum()
    k_pred_pos = (key_df["y_pred"] == 1).sum()
    k_caught = ((key_df["y_true"] == 1) & (key_df["y_pred"] == 1)).sum()
    k_recall = k_caught / k_true if k_true else 0.0
    k_precision = k_caught / k_pred_pos if k_pred_pos else 0.0

    # Khartoum calculations
    kh_true = kh_df["y_true"].sum()
    kh_pred_pos = (kh_df["y_pred"] == 1).sum()
    kh_caught = ((kh_df["y_true"] == 1) & (kh_df["y_pred"] == 1)).sum()
    kh_recall = kh_caught / kh_true if kh_true else 0.0
    kh_precision = kh_caught / kh_pred_pos if kh_pred_pos else 0.0

    stability_rows.append({
        "Seed": seed,
        "Key Regions Caught": f"{k_caught}/{k_true}",
        "Key Regions Recall": f"{k_recall * 100:.1f}%",
        "Key Regions Precision": f"{k_precision * 100:.1f}%",
        "Khartoum Caught": f"{kh_caught}/{kh_true}",
        "Khartoum Recall": f"{kh_recall * 100:.1f}%",
        "Khartoum Precision": f"{kh_precision * 100:.1f}%",
    })

regional_stability_table = pd.DataFrame(stability_rows)
regional_stability_table

# Summary INCORRECT 

Pulling the findings together across all evaluation windows:

* **Train-CV:** None of the text models outperform the baseline in terms of generalisation during training (2018–2022). Conflict-only text without PCA is the least weak of four underperforming options, beating baseline on F1 in 25% of matched pairs and 12.5% on AUPR. This performance gap is plausibly explained by the training period containing no civil war dynamics to learn from.

* **Onset (Aggregate):** Performance splits cleanly along corpus lines. Conflict-only text scores well on aggregate AUPR (beating baseline in 68.8–75.0% of matched pairs) but fails on F1 (12.5–25.0%), indicating a conservative operating point that ranks escalation months well overall but is too cautious at the deployed threshold. Conversely, all-event text without PCA dominates on F1 (87.5% better than baseline), prioritising recall over precision.

* **Onset (Regional):** Aggregate AUPR conceals geographic failures. Conflict-only text variants fail in the real-life scenario, completely missing the Khartoum outbreak (0/5) and catching at most 4 of 27 key regional escalation months. All-event text without PCA is the only model that catches fighting where it actually occurred (24/27 key regions, 2/5 Khartoum), though this high recall comes at the cost of a lower precision level.

* **Active Conflict:** The baseline wins clearly for three of the four text variants once active war is underway, and decisively so for conflict-only non-PCA. All-event text with PCA is the sole exception, running roughly level with or marginally ahead of baseline, demonstrating that corpus breadth combined with dimensionality reduction is required to maintain signal during active conflict.


Given the project's objective is to evaluate whether adding text improves conflict escalation predictions, **all-event text without PCA** remains the strongest candidate for onset detection specifically, as it is the only variant that successfully flags real-world escalation hubs like Khartoum. For ongoing active conflict monitoring, **all-event text with PCA** is worth retaining, as it is the only text model that avoids performance decay once conflict is underway. Conflict-only text, in either PCA or non-PCA form, is the weakest performer across all three periods and is hardest to justify keeping in the final model.